# MammoDiffusion LDM - 04b extra1361

Notebook finale per rilanciare e tenere ordinato l'esperimento LDM Keras prodotto da Alex.

- I dati reali preprocessati restano condivisi in `data/processed`.
- I dati aumentati tradizionali restano condivisi in `data/real_augmented`.
- Gli artefatti dell'esperimento vanno in `experiments/diffusers/06_ldm_extra1361_fromscratch`.
- Il training pesante vive in `notebooks/utility/train_ldm_v2.py`.


## 1. Selezione GPU


In [ ]:
# === Bootstrap unificato notebooks/ ===
# Funziona dalla root del progetto e da ogni sottocartella della struttura notebooks/.
import sys as _sys
from pathlib import Path as _Path


def _find_mammo_root():
    for _candidate in [_Path.cwd().resolve(), *_Path.cwd().resolve().parents]:
        if _candidate.name == "MammoDiffusion":
            return _candidate
        if (_candidate / "data").is_dir() and (_candidate / "notebooks").is_dir():
            return _candidate
    raise FileNotFoundError("Root MammoDiffusion non trovata da " + str(_Path.cwd()))


PROJECT_ROOT = _find_mammo_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
UTILITY_DIR = NOTEBOOKS_DIR / "utility"
for _path in (str(UTILITY_DIR), str(NOTEBOOKS_DIR)):
    if _path not in _sys.path:
        _sys.path.insert(0, _path)

BASE = PROJECT_ROOT
BASE_DIR = PROJECT_ROOT
BASE_PATH = str(PROJECT_ROOT) + "/"
# === Fine bootstrap unificato ===

import os
import subprocess
import sys
from pathlib import Path

CUDA_ROOT = Path(os.environ.get("MAMMODIFFUSION_CUDA_ROOT", os.environ.get("CONDA_PREFIX", sys.prefix)))


libdevice_path = CUDA_ROOT / "nvvm" / "libdevice" / "libdevice.10.bc"
if libdevice_path.exists():
    os.environ["XLA_FLAGS"] = f"--xla_gpu_cuda_data_dir={CUDA_ROOT}"
else:
    print("libdevice.10.bc non trovato in:", libdevice_path)

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.stdout.strip():
        print("GPU fisiche disponibili:")
        print(result.stdout.strip())
except FileNotFoundError:
    print("nvidia-smi non disponibile in questo ambiente.")

print("XLA_FLAGS:", os.environ.get("XLA_FLAGS", ""))
# GPU dedicata al training; la generazione usa invece GENERATION_GPU_DEVICES nei subprocess.
TRAIN_GPU_VISIBLE_DEVICES = "0"

def training_subprocess_env():
    env = os.environ.copy()
    if TRAIN_GPU_VISIBLE_DEVICES is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(TRAIN_GPU_VISIBLE_DEVICES)
    return env

# Generazione multi-GPU (non influenza il training).
PARALLEL_GENERATION = True
GENERATION_GPU_DEVICES = "auto"
GENERATION_MAX_WORKERS = None

def add_generation_parallel_args(command):
    if PARALLEL_GENERATION:
        command.extend(["--generation-gpus", GENERATION_GPU_DEVICES])
        if GENERATION_MAX_WORKERS is not None:
            command.extend(["--max-generation-workers", str(GENERATION_MAX_WORKERS)])
    else:
        command.extend(["--generation-gpus", "off"])
    return command


## 2. Setup


In [ ]:
# L'ambiente tf-gpu del progetto include gia' le dipendenze.
# Impostare True solo quando si prepara intenzionalmente un ambiente nuovo.
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    %pip install -q --upgrade pip
    %pip install -q pandas numpy matplotlib scikit-learn pillow gdown tensorflow scikit-image scipy psutil codecarbon torch torchvision torchmetrics torch-fidelity prdc
else:
    print('Dipendenze gia presenti: installazione saltata.')


## 3. Path progetto ed esperimento


In [ ]:
from pathlib import Path
import shutil
import sys


PROJECT_NAME = "MammoDiffusion"
EXPERIMENT_NAME = "diffusers/06_ldm_extra1361_fromscratch"
RESULTS_STAGE_NAME = "diffusers/06_ldm_extra1361_fromscratch"

# Se serve su Colab/Drive:
# PROJECT_ROOT_OVERRIDE = Path("/content/drive/MyDrive/MammoDiffusion")
PROJECT_ROOT_OVERRIDE = None


def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    if override is not None:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE non esiste: {root}")
        return root

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == project_name:
            return candidate
        has_notebooks = (candidate / "notebooks").exists() or (candidate / "notebooks").exists()
        if ((candidate / "data").exists() and has_notebooks) or ((candidate / ".git").exists() and has_notebooks):
            return candidate

    for candidate in [
        cwd / project_name,
        Path("/content") / project_name,
        Path("/content/drive/MyDrive") / project_name,
        Path.home() / project_name,
    ]:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Non riesco a trovare la root MammoDiffusion. "
        "Esegui il notebook dalla repo o imposta PROJECT_ROOT_OVERRIDE."
    )


PROJECT_ROOT = find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / ("notebooks" if (PROJECT_ROOT / "notebooks").is_dir() else "notebooks")
UTILITY_DIR = NOTEBOOKS_DIR / "utility" if (NOTEBOOKS_DIR / "utility").is_dir() else NOTEBOOKS_DIR
DATA_DIR = PROJECT_ROOT / "data"
DATA_PROCESSED_DIR = DATA_DIR / "processed"
ARCHIVES_DIR = DATA_DIR / "archives"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / EXPERIMENT_NAME
SOURCE_EXPERIMENT_DIR = PROJECT_ROOT / "experiments/diffusers/05_ldm_basic_fromscratch"

MODELS_DIR = EXPERIMENT_DIR / "models"
CHECKPOINTS_DIR = SOURCE_EXPERIMENT_DIR / "checkpoints_ldm"
LATENTS_DIR = EXPERIMENT_DIR / "latents"
LOGS_DIR = EXPERIMENT_DIR / "logs"
RESULTS_DIR = PROJECT_ROOT / "results" / RESULTS_STAGE_NAME
RESULTS_PLOTS_DIR = RESULTS_DIR / "plots"
RESULTS_METRICS_DIR = RESULTS_DIR / "metrics"
RESULTS_ECOTRACKER_DIR = RESULTS_DIR / "ecotracker"
HELPER_PATH = UTILITY_DIR / "train_ldm_v2.py"
VAE_HELPER_PATH = UTILITY_DIR / "train_vae_v2.py"

for directory in [
    DATA_PROCESSED_DIR,
    ARCHIVES_DIR,
    EXPERIMENT_DIR,
    MODELS_DIR,
    LATENTS_DIR,
    LOGS_DIR,
    RESULTS_DIR,
    RESULTS_PLOTS_DIR,
    RESULTS_METRICS_DIR,
    RESULTS_ECOTRACKER_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PROCESSED_DIR:", DATA_PROCESSED_DIR)
print("EXPERIMENT_DIR:", EXPERIMENT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("HELPER_PATH:", HELPER_PATH)
print("VAE_HELPER_PATH:", VAE_HELPER_PATH)

if not HELPER_PATH.exists():
    raise FileNotFoundError(f"Helper non trovato: {HELPER_PATH}")
if not VAE_HELPER_PATH.exists():
    raise FileNotFoundError(f"Helper VAE non trovato: {VAE_HELPER_PATH}")


In [ ]:
# IDEMPOTENT_PHASE_MODES_V1
TRAIN_MODE = "skip"       # 06 riusa il training canonico di 05
GENERATION_MODE = "auto"  # auto | run | skip
EVALUATION_MODE = "auto"  # auto | run | skip | recompute
FILTER_MODE = "auto"      # auto | run | skip | recompute
PLAN_ONLY = False
ALLOW_HEAVY_RETRAIN = False       # must be True for auto mode to retrain from scratch
ALLOW_FULL_REGENERATION = False   # must be True for auto mode to regenerate a full image set

from artifact_phase_planner import plan_experiment, print_plan, phase_should_run
PHASE_MODES = {"training": TRAIN_MODE, "generation": GENERATION_MODE,
               "evaluation": EVALUATION_MODE, "filter": FILTER_MODE}
ALLOW_FLAGS = {"training": ALLOW_HEAVY_RETRAIN, "generation": ALLOW_FULL_REGENERATION}
PHASE_PLAN = plan_experiment(EXPERIMENT_DIR, PHASE_MODES, ALLOW_FLAGS)
print_plan(PHASE_PLAN)


## 4. Dataset preprocessato condiviso


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

import pandas as pd

try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown


# ZIP condiviso: train/0, train/1, val/0, val/1, test/0, test/1, metadata/*.csv
PROCESSED_DRIVE_ID = "1qQral_BIBlMl0QN3PllJukdYTOmNGWr3"
PROCESSED_ZIP_PATH = ARCHIVES_DIR / "processed.zip"
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
EXPECTED_SPLITS = ["train", "val", "test"]
EXPECTED_LABELS = ["0", "1"]
REQUIRED_METADATA = ["all_processed.csv", "train.csv", "val.csv", "test.csv"]


def count_images(folder):
    folder = Path(folder)
    if not folder.is_dir():
        return 0
    return sum(
        1
        for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def get_split_label_counts(processed_dir):
    processed_dir = Path(processed_dir).resolve()
    rows = []
    for split in EXPECTED_SPLITS:
        for label in EXPECTED_LABELS:
            rows.append(
                {
                    "split": split,
                    "label": int(label),
                    "folder": str(processed_dir / split / label),
                    "n_images": count_images(processed_dir / split / label),
                }
            )
    return pd.DataFrame(rows)


def processed_dataset_ready(processed_dir):
    processed_dir = Path(processed_dir).resolve()
    if not processed_dir.exists():
        return False
    counts_df = get_split_label_counts(processed_dir)
    ready = (counts_df["n_images"] > 0).all()
    if not ready and any((processed_dir / split).exists() for split in EXPECTED_SPLITS):
        print("Dataset preprocessato trovato, ma struttura incompleta:")
        print(counts_df[["split", "label", "n_images"]].to_string(index=False))
    return bool(ready)


def metadata_complete(processed_dir):
    metadata_dir = Path(processed_dir) / "metadata"
    return all((metadata_dir / name).exists() for name in REQUIRED_METADATA)


def parse_filename_metadata(image_path):
    parts = Path(image_path).stem.split("_")
    patient_id = parts[0] if len(parts) >= 1 else Path(image_path).stem
    image_id = parts[1] if len(parts) >= 2 else Path(image_path).stem
    laterality = parts[2] if len(parts) >= 3 else "unknown"
    view = parts[3] if len(parts) >= 4 else "unknown"
    return patient_id, image_id, laterality, view


def rebuild_metadata_from_folders(processed_dir):
    processed_dir = Path(processed_dir).resolve()
    metadata_dir = processed_dir / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for split in EXPECTED_SPLITS:
        for label_name in EXPECTED_LABELS:
            label_dir = processed_dir / split / label_name
            label = int(label_name)
            for image_path in sorted(label_dir.rglob("*")):
                if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue
                patient_id, image_id, laterality, view = parse_filename_metadata(image_path)
                rows.append(
                    {
                        "patient_id": str(patient_id),
                        "image_id": str(image_id),
                        "laterality": str(laterality),
                        "view": str(view),
                        "label": label,
                        "cancer": label,
                        "patient_label": label,
                        "split": split,
                        "source": "real",
                        "original_path": "",
                        "processed_path": str(image_path),
                    }
                )
    if not rows:
        raise FileNotFoundError(f"Nessuna immagine valida trovata in {processed_dir}")

    df = pd.DataFrame(rows).sort_values(
        ["split", "label", "patient_id", "image_id"]
    ).reset_index(drop=True)
    df.to_csv(metadata_dir / "all_processed.csv", index=False)
    for split in EXPECTED_SPLITS:
        df[df["split"] == split].reset_index(drop=True).to_csv(
            metadata_dir / f"{split}.csv",
            index=False,
        )
    print("Metadata CSV ricostruiti in:", metadata_dir)
    print(pd.crosstab(df["split"], df["label"]))


def ensure_metadata(processed_dir):
    if metadata_complete(processed_dir):
        print("Metadata CSV gia' presenti:", Path(processed_dir) / "metadata")
        return
    print("Metadata CSV mancanti: li ricostruisco da train/val/test/0-1.")
    rebuild_metadata_from_folders(processed_dir)


def download_processed_zip():
    if PROCESSED_ZIP_PATH.exists():
        if zipfile.is_zipfile(PROCESSED_ZIP_PATH):
            print("Archivio gia' presente, salto download:", PROCESSED_ZIP_PATH)
            return
        print("Archivio presente ma non valido: lo riscarico.")
        PROCESSED_ZIP_PATH.unlink()
    print("Download di processed.zip da Google Drive...")
    gdown.download(id=PROCESSED_DRIVE_ID, output=str(PROCESSED_ZIP_PATH), quiet=False)

    if not PROCESSED_ZIP_PATH.exists() or PROCESSED_ZIP_PATH.stat().st_size == 0:
        raise RuntimeError("Download fallito. Controlla la condivisione del file Drive.")
    if not zipfile.is_zipfile(PROCESSED_ZIP_PATH):
        raise RuntimeError(f"Il file scaricato non e' uno ZIP valido: {PROCESSED_ZIP_PATH}")
    print("Download completato:", PROCESSED_ZIP_PATH)


def clear_processed_dir():
    DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    for item in DATA_PROCESSED_DIR.iterdir():
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()


def extract_processed_zip():
    print("Estrazione di processed.zip dentro:", DATA_PROCESSED_DIR)
    clear_processed_dir()
    with zipfile.ZipFile(PROCESSED_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(DATA_PROCESSED_DIR)


def prepare_processed_dataset():
    if processed_dataset_ready(DATA_PROCESSED_DIR):
        print("Dataset preprocessato gia' presente: salto download ed estrazione.")
        ensure_metadata(DATA_PROCESSED_DIR)
        return DATA_PROCESSED_DIR.resolve()

    PROCESSED_ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    download_processed_zip()
    extract_processed_zip()

    if not processed_dataset_ready(DATA_PROCESSED_DIR):
        counts_df = get_split_label_counts(DATA_PROCESSED_DIR)
        print(counts_df[["split", "label", "folder", "n_images"]].to_string(index=False))
        raise FileNotFoundError(
            "processed.zip estratto, ma la struttura non e' quella attesa. "
            "Lo ZIP deve contenere direttamente train/0, train/1, val/0, val/1, "
            "test/0, test/1 e metadata/*.csv."
        )

    ensure_metadata(DATA_PROCESSED_DIR)
    return DATA_PROCESSED_DIR.resolve()


DATASET_ROOT = prepare_processed_dataset()
print("DATASET_ROOT:", DATASET_ROOT)
print(get_split_label_counts(DATASET_ROOT)[["split", "label", "n_images"]].to_string(index=False))


## 5. Training VAE


In [ ]:
# Nessun training duplicato: 06 estende il pool generato dal modello di 05.
if not any(CHECKPOINTS_DIR.glob('ldm_unet_*.keras')):
    raise FileNotFoundError(
        f"Checkpoint condivisi non trovati: {CHECKPOINTS_DIR}. "
        "Esegui prima 05_LDM_Basic_FromScratch.ipynb."
    )
print("Checkpoint condivisi da 05:", CHECKPOINTS_DIR)


### Visualizzazione training VAE (sola lettura)

La cella seguente legge solo i plot gia' salvati su disco da `train_vae_v2.py` (`vae_metrics.png`, `vae_reconstruction.png`): non rilancia training.

In [ ]:
from IPython.display import display
from PIL import Image as PILImage

for plot_name in ["vae_metrics.png", "vae_reconstruction.png"]:
    plot_path = RESULTS_PLOTS_DIR / plot_name
    if plot_path.exists():
        img = PILImage.open(plot_path)
        if plot_name == "vae_reconstruction.png":
            # Mostra solo le prime 3 ricostruzioni (la griglia salvata ha 6 colonne)
            w, h = img.size
            img = img.crop((0, 0, w // 2, h))
        display(img)
        print("Plot:", plot_path)
    else:
        print("Plot non trovato:", plot_path)

## 6. Lancio training helper


In [ ]:
# Nessun training duplicato: 06 estende il pool generato dal modello di 05.
if not any(CHECKPOINTS_DIR.glob('ldm_unet_*.keras')):
    raise FileNotFoundError(
        f"Checkpoint condivisi non trovati: {CHECKPOINTS_DIR}. "
        "Esegui prima 05_LDM_Basic_FromScratch.ipynb."
    )
print("Checkpoint condivisi da 05:", CHECKPOINTS_DIR)


### Visualizzazione training LDM (sola lettura)

La cella seguente legge solo il plot gia' salvato su disco da `train_ldm_v2.py` (`ldm_metrics.png`): non rilancia training.

In [ ]:
from IPython.display import display
from PIL import Image as PILImage

plot_path = RESULTS_PLOTS_DIR / "ldm_metrics.png"
if plot_path.exists():
    display(PILImage.open(plot_path))
    print("Plot:", plot_path)
else:
    print("Plot non trovato:", plot_path)

## 7. Valutazione checkpoint

Il tracking di sostenibilita' viene attivato passando `--eco-track` a `evaluate_ldm_v2.py`. I record EcoTracker vengono aggiunti a `results/2_diffusers/06_ldm_extra1361_fromscratch/ecotracker/ldm_evaluation_ecotracker.jsonl` al completamento di ciascuno stage misurato, non continuamente durante lo stage.


Il best checkpoint viene selezionato minimizzando il FID delle immagini positive,
poiché la generazione finale è destinata all’aumento della classe tumorale.
Le metriche della classe negativa vengono comunque monitorate per controllare
la qualità globale del modello, ma non determinano la selezione principale.


In [ ]:
# IDEMPOTENT_GUARD_V1:evaluation
# Keep configuration available to downstream cells even when this phase is skipped.
EVAL_MIN_STEP = 1_000
N_GEN_PER_CLASS = 100
EVAL_SAMPLE_STEPS = 100
EVAL_GUIDANCE_SCALE = 1.5
EVAL_MINI_BATCH = 1
EVAL_INCEPTION_BATCH = 8
EVAL_INCEPTION_WEIGHTS = "imagenet"
EVAL_FORCE_RECOMPUTE = False
EVAL_MODE = "both"
EVAL_DECODE_ON_CPU = False
EVAL_ECO_TRACK = True

if phase_should_run(PHASE_PLAN, "evaluation", PLAN_ONLY):
    import os
    import subprocess
    import time

    eval_log_path = LOGS_DIR / "ldm_evaluate.log"
    eval_cmd = [
        sys.executable,
        str(UTILITY_DIR / "evaluate_ldm_v2.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--checkpoints-dir", str(CHECKPOINTS_DIR),
        "--mode", EVAL_MODE,
        "--min-step", str(EVAL_MIN_STEP),
        "--n-gen-per-class", str(N_GEN_PER_CLASS),
        "--sample-steps", str(EVAL_SAMPLE_STEPS),
        "--guidance-scale", str(EVAL_GUIDANCE_SCALE),
        "--mini-batch", str(EVAL_MINI_BATCH),
        "--inception-batch", str(EVAL_INCEPTION_BATCH),
        "--inception-weights", EVAL_INCEPTION_WEIGHTS,
        "--results-stage-name", RESULTS_STAGE_NAME,
    ]

    if EVAL_FORCE_RECOMPUTE:
        eval_cmd.append("--force-recompute")
    if EVAL_DECODE_ON_CPU:
        eval_cmd.append("--decode-on-cpu")
    if EVAL_ECO_TRACK:
        eval_cmd.append("--eco-track")

    add_generation_parallel_args(eval_cmd)
    env = os.environ.copy()
    print("Comando evaluation:")
    print(" ".join(eval_cmd))
    print("Log:", eval_log_path)
    print("CUDA_VISIBLE_DEVICES:", env.get("CUDA_VISIBLE_DEVICES", ""))
    print("XLA_FLAGS:", env.get("XLA_FLAGS", ""))

    with open(eval_log_path, "w", encoding="utf-8") as log_file:
        proc_eval = subprocess.Popen(
            eval_cmd,
            cwd=str(PROJECT_ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )

    pid_eval = proc_eval.pid
    print(f"Evaluation avviata - PID {pid_eval}")

    with open(eval_log_path, "r", encoding="utf-8", errors="replace") as log_file:
        while proc_eval.poll() is None:
            line = log_file.readline()
            if line:
                print(line, end="", flush=True)
            else:
                time.sleep(0.5)

        for line in log_file:
            print(line, end="", flush=True)

    print("Processo evaluation terminato con return code:", proc_eval.returncode)
    if proc_eval.returncode != 0:
        raise RuntimeError(f"Evaluation fallita. Controlla il log: {eval_log_path}")

### Trend PRDC per checkpoint lungo lo sweep (sola lettura)

Legge `evaluation/sweep_results.csv` (appena calcolato dalla cella precedente) e plotta precision/recall/density/coverage per ogni checkpoint, classe 0 e classe 1, con il checkpoint migliore evidenziato. Permette di vedere se le metriche migliorano in modo monotono o se c'e' un crollo (mode collapse) in qualche fase del training.

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sweep_df = (
    pd.read_csv(EXPERIMENT_DIR / "evaluation" / "sweep_results.csv")
    .sort_values("checkpoint_order")
    .reset_index(drop=True)
)
with open(EXPERIMENT_DIR / "evaluation" / "best_checkpoint.json", encoding="utf-8") as handle:
    best_ckpt = json.load(handle)

best_id = best_ckpt["best_checkpoint_id"]
x = np.arange(len(sweep_df))
labels = sweep_df["checkpoint_id"].astype(str).tolist()
best_positions = sweep_df.index[sweep_df["checkpoint_id"].astype(str) == best_id].tolist()
best_x = best_positions[0] if best_positions else None

metric_specs = [
    ("precision_0", "precision_1", "Precision"),
    ("recall_0", "recall_1", "Recall"),
    ("density_0", "density_1", "Density"),
    ("coverage_0", "coverage_1", "Coverage"),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
for ax, (col_0, col_1, title) in zip(axes.ravel(), metric_specs):
    ax.plot(x, sweep_df[col_0].astype(float), marker="o", linewidth=1.8, label="classe 0")
    ax.plot(x, sweep_df[col_1].astype(float), marker="s", linewidth=1.8, label="classe 1")
    if best_x is not None:
        ax.axvline(best_x, color="crimson", linestyle="--", linewidth=1.4)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="best")

fig.suptitle(f"Trend PRDC per checkpoint lungo lo sweep - BEST: {best_id}", fontsize=14)
output_path = RESULTS_PLOTS_DIR / "checkpoint_prdc_comparison.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()

print("Salvato:", output_path)

## 8. Generazione sintetiche e reverse diffusion

La generazione finale usa `--eco-track` su `generate_ldm_v2.py`. I record EcoTracker vengono aggiunti a `results/2_diffusers/06_ldm_extra1361_fromscratch/ecotracker/ldm_pipeline_ecotracker.jsonl` al completamento degli stage `ldm_generate_raw`, `ldm_filter_raw` e `ldm_validate_raw_filtered`, non continuamente durante lo stage.


In [ ]:
# IDEMPOTENT_GUARD_V1:generation
# Keep configuration available to validation/final-evaluation cells when generation is skipped.
GEN_MODE = "both"
GEN_N_RAW = 4083  # 2722 gia' presenti dal primo esperimento + 1361 nuove raw
GEN_N_SELECTED = 1361
GEN_TARGET_LABEL = 1
GEN_BATCH_SIZE = 1
GEN_SAMPLE_STEPS = 100
GEN_GUIDANCE_SCALE = 1.5
GEN_MODEL_PATH = CHECKPOINTS_DIR / "ldm_unet_best_eval.keras"
GEN_ECO_TRACK = True

if phase_should_run(PHASE_PLAN, "generation", PLAN_ONLY):
    import os
    import subprocess
    import time

    gen_log_path = LOGS_DIR / "ldm_generate.log"
    gen_cmd = [
        sys.executable,
        str(UTILITY_DIR / "generate_ldm_v2.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--model-path", str(GEN_MODEL_PATH),
        "--mode", GEN_MODE,
        "--n-raw", str(GEN_N_RAW),
        "--n-selected", str(GEN_N_SELECTED),
        "--target-label", str(GEN_TARGET_LABEL),
        "--batch-size", str(GEN_BATCH_SIZE),
        "--sample-steps", str(GEN_SAMPLE_STEPS),
        "--guidance-scale", str(GEN_GUIDANCE_SCALE),
        "--results-stage-name", RESULTS_STAGE_NAME,
    ]

    if GEN_ECO_TRACK:
        gen_cmd.append("--eco-track")

    add_generation_parallel_args(gen_cmd)
    env = os.environ.copy()
    print("Comando generation:")
    print(" ".join(gen_cmd))
    print("Log:", gen_log_path)
    print("CUDA_VISIBLE_DEVICES:", env.get("CUDA_VISIBLE_DEVICES", ""))
    print("XLA_FLAGS:", env.get("XLA_FLAGS", ""))

    with open(gen_log_path, "w", encoding="utf-8") as log_file:
        proc_gen = subprocess.Popen(
            gen_cmd,
            cwd=str(PROJECT_ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )

    pid_gen = proc_gen.pid
    print(f"Generation avviata - PID {pid_gen}")

    with open(gen_log_path, "r", encoding="utf-8", errors="replace") as log_file:
        while proc_gen.poll() is None:
            line = log_file.readline()
            if line:
                print(line, end="", flush=True)
            else:
                time.sleep(0.5)

        for line in log_file:
            print(line, end="", flush=True)

    print("Processo generation terminato con return code:", proc_gen.returncode)
    if proc_gen.returncode != 0:
        raise RuntimeError(f"Generation fallita. Controlla il log: {gen_log_path}")

### Motivi di scarto del filtro adattivo (sola lettura)

Legge `reject_counts` da `synthetic_filter_summary.json` (appena prodotto dalla generazione/filtro qui sopra) e mostra quante immagini sono state scartate per ciascun motivo (`too_empty`, `too_full`, `low_contrast`, `fragmented_mask`, `touches_top_border`, `touches_bottom_border`).

In [ ]:
import json

import matplotlib.pyplot as plt

with open(RESULTS_METRICS_DIR / "synthetic_filter_summary.json", encoding="utf-8") as handle:
    filter_summary = json.load(handle)

reject_counts = filter_summary["reject_counts"]
n_raw = filter_summary["n_raw"]
n_accepted = filter_summary["n_accepted"]
n_selected = filter_summary["n_selected"]

reasons = list(reject_counts.keys())
counts = [reject_counts[reason] for reason in reasons]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(reasons, counts, color="#c0392b")
ax.bar_label(bars, padding=3)
ax.set_title(f"Motivi di scarto del filtro - {n_raw} raw, {n_accepted} accettate, {n_selected} selezionate")
ax.set_ylabel("Immagini scartate")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

output_path = RESULTS_PLOTS_DIR / "synthetic_filter_reject_reasons.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()

print("Salvato:", output_path)
print("Conteggi:", reject_counts)

### Campioni accettati/rifiutati e distribuzione punteggio (sola lettura)

Mostra `synthetic_filter_sample.png` (esempi accettati vs rifiutati) e `synthetic_filter_distribution.png` (istogramma dello score e confronto nonblack_ratio/std_nonblack vs reali), gia' generati da `adaptive_mammography_filter.py` durante il filtro qui sopra.

In [ ]:
from IPython.display import display
from PIL import Image as PILImage

for plot_name in ["synthetic_filter_sample.png", "synthetic_filter_distribution.png"]:
    plot_path = RESULTS_PLOTS_DIR / plot_name
    if plot_path.exists():
        display(PILImage.open(plot_path))
        print("Plot:", plot_path)
    else:
        print("Plot non trovato:", plot_path)

### Confronto raw vs filtrate (richiede un nuovo calcolo)

Lancia `generate_ldm_v2.py --mode validate`: confronta sul **validation set** `raw_complete`, `raw_balanced_seed42` (stessa numerosita' delle filtrate) e `filtered`, con lo stesso backend (`generative_evaluator.py`) gia' usato per le altre metriche. Non incluso in `--mode both` qui sopra: e' un calcolo aggiuntivo (qualche minuto in piu', su GPU; piu' lungo su CPU).

In [ ]:
# IDEMPOTENT_GUARD_V1:generation
if phase_should_run(PHASE_PLAN, "generation", PLAN_ONLY):
    import os
    import subprocess
    import time

    VALIDATE_N_RAW = GEN_N_RAW
    VALIDATE_N_SELECTED = GEN_N_SELECTED
    VALIDATE_TARGET_LABEL = GEN_TARGET_LABEL
    VALIDATE_BALANCED_SEED = 42
    VALIDATE_INCEPTION_BATCH = EVAL_INCEPTION_BATCH
    VALIDATE_IS_SPLITS = 10
    VALIDATE_KNN_K = 3
    VALIDATE_ECO_TRACK = True

    validate_log_path = LOGS_DIR / "ldm_validate.log"
    validate_cmd = [
        sys.executable,
        str(UTILITY_DIR / "generate_ldm_v2.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--mode", "validate",
        "--n-raw", str(VALIDATE_N_RAW),
        "--n-selected", str(VALIDATE_N_SELECTED),
        "--target-label", str(VALIDATE_TARGET_LABEL),
        "--balanced-seed", str(VALIDATE_BALANCED_SEED),
        "--inception-batch", str(VALIDATE_INCEPTION_BATCH),
        "--is-splits", str(VALIDATE_IS_SPLITS),
        "--knn-k", str(VALIDATE_KNN_K),
        "--results-stage-name", RESULTS_STAGE_NAME,
    ]
    if VALIDATE_ECO_TRACK:
        validate_cmd.append("--eco-track")

    add_generation_parallel_args(validate_cmd)
    env = os.environ.copy()
    print("Comando validate:")
    print(" ".join(validate_cmd))
    print("Log:", validate_log_path)

    with open(validate_log_path, "w", encoding="utf-8") as log_file:
        proc_validate = subprocess.Popen(
            validate_cmd,
            cwd=str(PROJECT_ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )

    pid_validate = proc_validate.pid
    print(f"Validate avviato - PID {pid_validate}")

    with open(validate_log_path, "r", encoding="utf-8", errors="replace") as log_file:
        while proc_validate.poll() is None:
            line = log_file.readline()
            if line:
                print(line, end="", flush=True)
            else:
                time.sleep(0.5)
        for line in log_file:
            print(line, end="", flush=True)

    print("Processo validate terminato con return code:", proc_validate.returncode)
    if proc_validate.returncode != 0:
        raise RuntimeError(f"Validate fallito. Controlla il log: {validate_log_path}")

### Visualizzazione raw vs filtrate (sola lettura)

Legge `raw_vs_filtered_validation.csv` prodotto dalla cella precedente e costruisce gli stessi grafici gia' presenti per il collega in `02_SD21_Filtered_100steps.ipynb` (`raw_vs_filtered_fid.png`, `filter_before_after.png`, `filter_prdc_radar.png`, `filter_oriented_percent_delta.png`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

raw_vs_filtered_candidates = [
    RESULTS_METRICS_DIR / "positive" / "raw_vs_filtered_validation.csv",
    RESULTS_METRICS_DIR / "raw_vs_filtered_validation.csv",  # retained compatibility metric
]
raw_vs_filtered_path = next((path for path in raw_vs_filtered_candidates if path.exists()), None)
assert raw_vs_filtered_path is not None, (
    f"File non trovato in nessuno dei percorsi attesi: {raw_vs_filtered_candidates}. "
    "Esegui prima la cella di validate."
)

df_rvf = pd.read_csv(raw_vs_filtered_path).set_index("dataset")
balanced_name = next(name for name in df_rvf.index if name.startswith("raw_balanced"))

# 1. FID: raw completo vs filtrate
fid_order = ["raw_complete", "filtered"]
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(fid_order, df_rvf.loc[fid_order, "FID"], color=["#777777", "#2e8b57"])
ax.set_title("FID sul validation set - raw completo vs filtrate")
ax.set_ylabel("FID")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
output_path = RESULTS_PLOTS_DIR / "raw_vs_filtered_fid.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print("Salvato:", output_path)

raw_row = df_rvf.loc[balanced_name]
filtered_row = df_rvf.loc["filtered"]


def metric_pair(metric):
    return float(raw_row[metric]), float(filtered_row[metric])


# 2. Prima/dopo: ogni metrica mantiene la propria scala
metric_specs = [
    ("FID", "FID - più basso è meglio"),
    ("IS_mean", "Inception Score - più alto indica maggiore varietà"),
    ("precision", "Precision PRDC - più alto è meglio"),
    ("recall", "Recall PRDC - più alto indica maggiore copertura"),
]
fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)
for axis, (metric, title) in zip(axes.flat, metric_specs):
    raw_value, filtered_value = metric_pair(metric)
    delta = (filtered_value - raw_value) / raw_value * 100
    axis.plot([0, 1], [raw_value, filtered_value], color="#1f77b4", marker="o", linewidth=2.2, markersize=7)
    axis.annotate(
        "",
        xy=(1, filtered_value),
        xytext=(0, raw_value),
        arrowprops={"arrowstyle": "-|>", "color": "#1f77b4", "linewidth": 2.2, "mutation_scale": 13},
    )
    axis.annotate(
        f"{delta:+.1f}%",
        xy=(1, filtered_value),
        xytext=(7, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color="#1f77b4",
    )
    axis.set_xticks([0, 1], ["RAW (bilanciato)", "Filtrate"])
    axis.set_title(title)
    axis.grid(axis="y", alpha=0.25)
fig.suptitle("Effetto del filtro adattivo sull'LDM - classe positiva", fontsize=15)
output_path = RESULTS_PLOTS_DIR / "filter_before_after.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print("Salvato:", output_path)

# 3. Radar PRDC
prdc_metrics = ["precision", "recall", "density", "coverage"]
radar_angles = np.linspace(0, 2 * np.pi, len(prdc_metrics), endpoint=False).tolist()
radar_angles += radar_angles[:1]
raw_values = [float(raw_row[metric]) for metric in prdc_metrics]
filtered_values = [float(filtered_row[metric]) for metric in prdc_metrics]
radial_limit = min(0.5, max(raw_values + filtered_values) * 1.15)

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True}, constrained_layout=True)
for label, values, color in [("RAW (bilanciato)", raw_values, "#777777"), ("Filtrate", filtered_values, "#2e8b57")]:
    closed_values = values + values[:1]
    ax.plot(radar_angles, closed_values, color=color, linewidth=2, label=label)
    ax.fill(radar_angles, closed_values, color=color, alpha=0.18)
ax.set_xticks(radar_angles[:-1], [metric.capitalize() for metric in prdc_metrics])
ax.set_ylim(0, radial_limit)
ax.set_yticks(np.round(np.linspace(0, radial_limit, 6), 2))
ax.tick_params(axis="y", labelsize=8)
ax.set_title("Profilo PRDC prima e dopo il filtro adattivo - LDM classe positiva", pad=20)
ax.legend(loc="lower right", bbox_to_anchor=(1.25, -0.05))
output_path = RESULTS_PLOTS_DIR / "filter_prdc_radar.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print("Salvato:", output_path)

# 4. Delta percentuale orientato al miglioramento
delta_metrics = ["FID", "IS_mean", "precision", "recall", "density", "coverage"]
oriented_deltas = []
for metric in delta_metrics:
    raw_value, filtered_value = metric_pair(metric)
    raw_delta = (filtered_value - raw_value) / raw_value * 100
    oriented_deltas.append(-raw_delta if metric == "FID" else raw_delta)

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
y_positions = np.arange(len(delta_metrics))
colors = ["#2e8b57" if value >= 0 else "#c0392b" for value in oriented_deltas]
bars = ax.barh(y_positions, oriented_deltas, color=colors, edgecolor="white")

# Posiziona ogni label sempre verso l'esterno della barra (lontano dall'asse y),
# indipendentemente dal segno, cosi non si sovrappone mai ai tick label
for bar, value in zip(bars, oriented_deltas):
    width = bar.get_width()
    label = f"{value:+.1f}%"
    if width >= 0:
        x_pos = width
        ha = "left"
    else:
        x_pos = width
        ha = "right"
    ax.annotate(
        label,
        xy=(x_pos, bar.get_y() + bar.get_height() / 2),
        xytext=(3 if width >= 0 else -3, 0),
        textcoords="offset points",
        va="center",
        ha=ha,
        fontsize=9,
    )

# Espande i limiti dell'asse x per fare spazio alle label, evitando che vengano tagliate
x_min, x_max = ax.get_xlim()
margin = (x_max - x_min) * 0.08
ax.set_xlim(x_min - margin, x_max + margin)

ax.axvline(0, color="black", linewidth=1)
ax.set_yticks(y_positions, ["FID (segno invertito)", "IS_mean", "Precision", "Recall", "Density", "Coverage"])
ax.set_xlabel("Variazione percentuale orientata al miglioramento")
ax.set_title("Delta del filtro adattivo - LDM classe positiva")
ax.grid(axis="x", alpha=0.25)
ax.legend(
    handles=[
        Patch(facecolor="#2e8b57", label="Miglioramento"),
        Patch(facecolor="#c0392b", label="Riduzione / trade-off"),
    ],
    loc="best",
)
output_path = RESULTS_PLOTS_DIR / "filter_oriented_percent_delta.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print("Salvato:", output_path)

## 9. Metriche finali su filtrate selezionate vs test


In [ ]:
# IDEMPOTENT_GUARD_V1:filter
if phase_should_run(PHASE_PLAN, "filter", PLAN_ONLY):
    import os
    import subprocess
    import time

    FINAL_EVAL_TARGET_LABEL = GEN_TARGET_LABEL
    FINAL_EVAL_INCEPTION_BATCH = EVAL_INCEPTION_BATCH
    FINAL_EVAL_INCEPTION_WEIGHTS = EVAL_INCEPTION_WEIGHTS

    final_eval_log_path = LOGS_DIR / "ldm_final_filtered_eval.log"
    final_eval_cmd = [
        sys.executable,
        str(UTILITY_DIR / "evaluate_filtered_ldm_v2.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--target-label", str(FINAL_EVAL_TARGET_LABEL),
        "--inception-batch", str(FINAL_EVAL_INCEPTION_BATCH),
        "--inception-weights", FINAL_EVAL_INCEPTION_WEIGHTS,
        "--results-stage-name", RESULTS_STAGE_NAME,
    ]

    env = os.environ.copy()
    print("Comando final evaluation:")
    print(" ".join(final_eval_cmd))
    print("Log:", final_eval_log_path)
    print("CUDA_VISIBLE_DEVICES:", env.get("CUDA_VISIBLE_DEVICES", ""))
    print("XLA_FLAGS:", env.get("XLA_FLAGS", ""))

    with open(final_eval_log_path, "w", encoding="utf-8") as log_file:
        proc_final_eval = subprocess.Popen(
            final_eval_cmd,
            cwd=str(PROJECT_ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )

    pid_final_eval = proc_final_eval.pid
    print(f"Final evaluation avviata - PID {pid_final_eval}")

    with open(final_eval_log_path, "r", encoding="utf-8", errors="replace") as log_file:
        while proc_final_eval.poll() is None:
            line = log_file.readline()
            if line:
                print(line, end="", flush=True)
            else:
                time.sleep(0.5)

        for line in log_file:
            print(line, end="", flush=True)

    print("Processo final evaluation terminato con return code:", proc_final_eval.returncode)
    if proc_final_eval.returncode != 0:
        raise RuntimeError(f"Final evaluation fallita. Controlla il log: {final_eval_log_path}")


### Visualizzazione checkpoint e metriche finali sul test

La cella seguente legge solo artefatti già prodotti: mostra una griglia con una negativa e una positiva per ogni checkpoint, visualizza i plot comparativi dello sweep e crea un riepilogo grafico delle metriche finali calcolate sul test set.


In [ ]:
# Visualizzazione checkpoint e metriche finali sul test (sola lettura)
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image as PILImage


CHECKPOINT_PREVIEW_INDEX = 0
MAX_CHECKPOINTS_PER_PAGE = 20  # >= numero checkpoint: salta la suddivisione in pagine


def _resolve_results_dirs():
    """Usa le directory results già definite; se mancano, le crea con convenzione del notebook."""
    if "RESULTS_PLOTS_DIR" in globals():
        plots_dir = Path(RESULTS_PLOTS_DIR)
    elif "PLOTS_DIR" in globals():
        plots_dir = Path(PLOTS_DIR)
    else:
        stage_name = (
            "diffusers/06_ldm_extra1361_fromscratch"
            if "CHECKPOINTS_DIR" in globals()
            else "diffusers/02_sd21_filtered_100steps"
        )
        plots_dir = Path(PROJECT_ROOT) / "results" / stage_name / "plots"

    if "RESULTS_METRICS_DIR" in globals():
        metrics_dir = Path(RESULTS_METRICS_DIR)
    elif "METRICS_DIR" in globals():
        metrics_dir = Path(METRICS_DIR)
    else:
        metrics_dir = plots_dir.parent / "metrics"

    plots_dir.mkdir(parents=True, exist_ok=True)
    metrics_dir.mkdir(parents=True, exist_ok=True)
    return plots_dir, metrics_dir


RESULTS_PLOTS_DIR, RESULTS_METRICS_DIR = _resolve_results_dirs()


def _format_cfg_label(guidance_scale, sample_steps):
    gs_label = f"{float(guidance_scale):g}".replace(".", "p")
    return f"cfg_gs{gs_label}_st{int(sample_steps)}"


def _first_readable_image(directory, preview_index=0):
    """Preferisce l'indice deterministico richiesto; se manca/corrotto usa il primo PNG leggibile."""
    directory = Path(directory)
    preferred_names = [
        f"{preview_index:04d}.png",
        f"gen_{preview_index:04d}.png",
    ]
    candidates = [directory / name for name in preferred_names]
    candidates.extend(sorted(directory.glob("*.png")))

    seen = set()
    for path in candidates:
        if path in seen:
            continue
        seen.add(path)
        if not path.exists():
            continue
        try:
            with PILImage.open(path) as image:
                return path, image.convert("L").copy()
        except Exception as exc:
            warnings.warn(f"Immagine non leggibile, provo fallback: {path} ({exc})")
    return None, None


def _load_checkpoint_preview_records():
    """Adatta automaticamente i layout checkpoint di notebook 02 e 04."""
    # Notebook 02: results/02.../checkpoint_validation_metrics.json + eval_checkpoints/checkpoint-*/{negative,positive}
    has_sd_eval_layout = ("EVAL_METRICS_PATH" in globals() and "EVAL_DIR" in globals() and Path(EVAL_METRICS_PATH).exists() and Path(EVAL_DIR).exists())
    if has_sd_eval_layout:
        with Path(EVAL_METRICS_PATH).open(encoding="utf-8") as handle:
            payload = json.load(handle)

        best_id = None
        if "BEST_CHECKPOINT" in globals():
            best_id = Path(BEST_CHECKPOINT).name
        if best_id is None and payload:
            best_id = min(payload, key=lambda row: row.get("avg_FID", math.inf)).get("ckpt_name")

        records = []
        for row in sorted(payload, key=lambda item: int(item.get("step", 0))):
            checkpoint_id = row["ckpt_name"]
            checkpoint_dir = Path(EVAL_DIR) / checkpoint_id
            records.append({
                "checkpoint_id": checkpoint_id,
                "title": checkpoint_id.replace("checkpoint-", "ckpt "),
                "order": int(row.get("step", len(records))),
                "is_best": checkpoint_id == best_id,
                "negative_dir": checkpoint_dir / "negative",
                "positive_dir": checkpoint_dir / "positive",
            })
        return records, "checkpoint_samples_negative_positive.png"

    # Notebook 04: experiment/evaluation/checkpoint_metrics.json + sweep_generated/<ckpt>/<cfg>/class_{0,1}
    checkpoint_metrics_path = Path(EXPERIMENT_DIR) / "evaluation" / "checkpoint_metrics.json"
    if checkpoint_metrics_path.exists():
        with checkpoint_metrics_path.open(encoding="utf-8") as handle:
            payload = json.load(handle)

        config = payload.get("config", {})
        cfg_label = _format_cfg_label(
            config.get("guidance_scale", globals().get("EVAL_GUIDANCE_SCALE", 1.5)),
            config.get("sample_steps", globals().get("EVAL_SAMPLE_STEPS", 100)),
        )
        best_id = payload.get("selection", {}).get("best_checkpoint_id")
        sweep_dir = Path(EXPERIMENT_DIR) / "evaluation" / "sweep_generated"

        records = []
        for row in sorted(payload.get("checkpoints", []), key=lambda item: item.get("checkpoint_order", 0)):
            checkpoint_id = row["checkpoint_id"]
            checkpoint_dir = sweep_dir / checkpoint_id / cfg_label
            records.append({
                "checkpoint_id": checkpoint_id,
                "title": checkpoint_id.replace("step_", "step "),
                "order": int(row.get("checkpoint_order", len(records))),
                "is_best": checkpoint_id == best_id,
                "negative_dir": checkpoint_dir / "class_0",
                "positive_dir": checkpoint_dir / "class_1",
            })
        return records, "checkpoint_samples_negative_positive.png"

    warnings.warn("Metriche checkpoint non trovate: salto la griglia immagini per checkpoint.")
    return [], "checkpoint_samples_negative_positive.png"


def _draw_checkpoint_sample_grid(records, output_path, title_suffix="", page_index=None):
    n_checkpoints = len(records)
    if n_checkpoints == 0:
        return None

    fig_width = max(6.0, 2.2 * n_checkpoints)
    fig, axes = plt.subplots(
        2,
        n_checkpoints,
        figsize=(fig_width, 5.0),
        squeeze=False,
        constrained_layout=True,
    )

    fallbacks = []
    for col, record in enumerate(records):
        for row_index, (class_label, directory_key) in enumerate([
            ("Negativa", "negative_dir"),
            ("Positiva", "positive_dir"),
        ]):
            axis = axes[row_index, col]
            image_path, image = _first_readable_image(
                record[directory_key],
                preview_index=CHECKPOINT_PREVIEW_INDEX,
            )
            if image is None:
                axis.text(0.5, 0.5, "immagine\nmancante", ha="center", va="center", fontsize=8)
                axis.set_facecolor("#f2f2f2")
            else:
                axis.imshow(np.asarray(image), cmap="gray", vmin=0, vmax=255)
                expected = (
                    Path(record[directory_key]) / f"{CHECKPOINT_PREVIEW_INDEX:04d}.png",
                    Path(record[directory_key]) / f"gen_{CHECKPOINT_PREVIEW_INDEX:04d}.png",
                )
                if image_path not in expected:
                    fallbacks.append((record["checkpoint_id"], class_label, image_path.name))

            axis.set_xticks([])
            axis.set_yticks([])
            for spine in axis.spines.values():
                spine.set_visible(bool(record["is_best"]))
                spine.set_edgecolor("crimson")
                spine.set_linewidth(2.0)

            if col == 0:
                axis.set_ylabel(class_label, fontsize=11, fontweight="bold")

        title = record["title"]
        if record["is_best"]:
            title = f"{title}\nBEST"
        axes[0, col].set_title(
            title,
            fontsize=9,
            color="crimson" if record["is_best"] else "black",
            fontweight="bold" if record["is_best"] else "normal",
        )

    page_text = f" - pagina {page_index}" if page_index is not None else ""
    fig.suptitle(
        f"Immagini generate per checkpoint - indice {CHECKPOINT_PREVIEW_INDEX}{title_suffix}{page_text}",
        fontsize=13,
    )
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Salvato:", output_path)

    if fallbacks:
        print("Fallback immagine usati:")
        for checkpoint_id, class_label, filename in fallbacks[:12]:
            print(f"  {checkpoint_id} / {class_label}: {filename}")
        if len(fallbacks) > 12:
            print(f"  ... altri {len(fallbacks) - 12} fallback")
    return output_path


def plot_checkpoint_samples_negative_positive():
    records, output_name = _load_checkpoint_preview_records()
    if not records:
        return []

    output_path = RESULTS_PLOTS_DIR / output_name
    saved_paths = [_draw_checkpoint_sample_grid(records, output_path)]

    if len(records) > MAX_CHECKPOINTS_PER_PAGE:
        for page, start in enumerate(range(0, len(records), MAX_CHECKPOINTS_PER_PAGE), start=1):
            chunk = records[start:start + MAX_CHECKPOINTS_PER_PAGE]
            page_path = RESULTS_PLOTS_DIR / f"checkpoint_samples_negative_positive_page_{page:02d}.png"
            saved_paths.append(_draw_checkpoint_sample_grid(chunk, page_path, page_index=page))

    return [path for path in saved_paths if path is not None]


def _find_final_test_metrics_csv():
    candidates = []
    if "FINAL_TEST_METRICS_CSV" in globals():
        candidates.append(Path(FINAL_TEST_METRICS_CSV))
    candidates.extend([
        RESULTS_METRICS_DIR / "final_filtered_vs_test.csv",
        RESULTS_METRICS_DIR / "final_test_metrics.csv",
        Path(EXPERIMENT_DIR) / "evaluation" / "final_filtered_vs_test.csv",
    ])
    for path in candidates:
        if path.exists():
            return path
    return None


def plot_final_test_metrics():
    metrics_path = _find_final_test_metrics_csv()
    if metrics_path is None:
        warnings.warn("Metriche test non trovate: salto il plot finale sul test.")
        return None

    df = pd.read_csv(metrics_path)
    metrics = [metric for metric in ["FID", "IS_mean", "precision", "recall", "density", "coverage"] if metric in df.columns]
    if not metrics:
        warnings.warn(f"Nessuna metrica attesa trovata in {metrics_path}")
        return None

    if "class" in df.columns:
        label_col = "class"
    elif "set_name" in df.columns:
        label_col = "set_name"
    else:
        label_col = "variant"
        df[label_col] = ["filtered_vs_test"] * len(df)

    plot_df = df[[label_col, *metrics]].copy()
    if "IS_std" in df.columns and "IS_std" not in plot_df.columns:
        plot_df["IS_std"] = df["IS_std"]
    plot_df[label_col] = plot_df[label_col].astype(str)

    output_path = RESULTS_PLOTS_DIR / "final_filtered_vs_test_metrics.png"
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
    axes = axes.ravel()

    if "FID" in metrics:
        plot_df.plot.bar(x=label_col, y="FID", ax=axes[0], color="#4C78A8", legend=False)
        axes[0].set_title("FID sul test - più basso è meglio")
        axes[0].set_xlabel("")
        axes[0].set_ylabel("FID")
        axes[0].grid(axis="y", alpha=0.25)
    else:
        axes[0].axis("off")

    if "IS_mean" in metrics:
        yerr = plot_df["IS_std"] if "IS_std" in df.columns else None
        plot_df.plot.bar(x=label_col, y="IS_mean", yerr=yerr, ax=axes[1], color="#F58518", legend=False, capsize=3)
        axes[1].set_title("Inception Score sul test")
        axes[1].set_xlabel("")
        axes[1].set_ylabel("IS mean")
        axes[1].grid(axis="y", alpha=0.25)
    else:
        axes[1].axis("off")

    prdc_metrics = [metric for metric in ["precision", "recall", "density", "coverage"] if metric in metrics]
    if prdc_metrics:
        x = np.arange(len(plot_df))
        width = 0.8 / max(1, len(prdc_metrics))
        colors = ["#54A24B", "#E45756", "#72B7B2", "#B279A2"]
        for index, metric in enumerate(prdc_metrics):
            axes[2].bar(
                x + (index - (len(prdc_metrics) - 1) / 2) * width,
                plot_df[metric].astype(float),
                width=width,
                label=metric,
                color=colors[index % len(colors)],
            )
        axes[2].set_title("PRDC sul test")
        axes[2].set_xticks(x, plot_df[label_col], rotation=45, ha="right")
        axes[2].set_ylabel("valore")
        axes[2].grid(axis="y", alpha=0.25)
        axes[2].legend(ncol=2, fontsize=8)
    else:
        axes[2].axis("off")

    table_metrics = [metric for metric in ["FID", "IS_mean", "precision", "recall", "density", "coverage"] if metric in metrics]
    table_df = plot_df[[label_col, *table_metrics]].copy()
    for metric in table_metrics:
        table_df[metric] = pd.to_numeric(table_df[metric], errors="coerce").round(4)
    axes[3].axis("off")
    axes[3].set_title("Metriche finali test")
    table = axes[3].table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        loc="center",
        cellLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1.0, 1.25)

    fig.suptitle("Valutazione finale sul test - configurazione congelata", fontsize=14)
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Salvato:", output_path)
    print("Metriche lette da:", metrics_path)
    return output_path


def display_checkpoint_metric_plots():
    plot_names = [
        "checkpoint_fid_comparison.png",
        "checkpoint_is_comparison.png",
        "best_checkpoint_summary.png",
    ]
    shown = []
    for plot_name in plot_names:
        plot_path = RESULTS_PLOTS_DIR / plot_name
        if not plot_path.exists():
            warnings.warn(f"Plot checkpoint non trovato: {plot_path}")
            continue
        display(PILImage.open(plot_path))
        print("Plot:", plot_path)
        shown.append(plot_path)
    return shown


checkpoint_plot_paths = plot_checkpoint_samples_negative_positive()
checkpoint_metric_plot_paths = display_checkpoint_metric_plots()
final_test_plot_path = plot_final_test_metrics()


# Visualizzazione Metriche filtered vs test sia per classe positiva che negativa

In [ ]:
# IDEMPOTENT_GUARD_V1:filter
if phase_should_run(PHASE_PLAN, "filter", PLAN_ONLY):
    NEG_LABEL = 0
    NEG_FILTERED_DIR = PROJECT_ROOT / "data" / "synthetic" / "06_ldm_extra1361_fromscratch" / "negative"
    NEG_EVAL_DIR = EXPERIMENT_DIR / "evaluation" / "negative"

    NEG_FINAL_EVAL_INCEPTION_BATCH = EVAL_INCEPTION_BATCH
    NEG_FINAL_EVAL_INCEPTION_WEIGHTS = EVAL_INCEPTION_WEIGHTS

    neg_final_eval_log_path = LOGS_DIR / "ldm_final_filtered_eval_negative.log"
    neg_final_eval_cmd = [
        sys.executable,
        str(UTILITY_DIR / "evaluate_filtered_ldm_v2.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--target-label", str(NEG_LABEL),
        "--synthetic-dir", str(NEG_FILTERED_DIR),
        "--inception-batch", str(NEG_FINAL_EVAL_INCEPTION_BATCH),
        "--inception-weights", NEG_FINAL_EVAL_INCEPTION_WEIGHTS,
        "--results-stage-name", RESULTS_STAGE_NAME,
    ]

    env = os.environ.copy()
    print("Comando final evaluation (negativi):")
    print(" ".join(neg_final_eval_cmd))
    print("Log:", neg_final_eval_log_path)

    with open(neg_final_eval_log_path, "w", encoding="utf-8") as log_file:
        proc_neg_final_eval = subprocess.Popen(
            neg_final_eval_cmd,
            cwd=str(PROJECT_ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )

    pid_neg_final_eval = proc_neg_final_eval.pid
    print(f"Final evaluation (negativi) avviata - PID {pid_neg_final_eval}")

    with open(neg_final_eval_log_path, "r", encoding="utf-8", errors="replace") as log_file:
        while proc_neg_final_eval.poll() is None:
            line = log_file.readline()
            if line:
                print(line, end="", flush=True)
            else:
                time.sleep(0.5)
        for line in log_file:
            print(line, end="", flush=True)

    print("Processo final evaluation (negativi) terminato con return code:", proc_neg_final_eval.returncode)
    if proc_neg_final_eval.returncode != 0:
        raise RuntimeError(f"Final evaluation (negativi) fallita. Controlla il log: {neg_final_eval_log_path}")

    # Confronto Positivo vs Negativo - solo immagini filtrate (1361 vs 1361) vs test
    df_pos = pd.read_csv(RESULTS_METRICS_DIR / "positive" / "final_filtered_vs_test.csv")
    df_neg = pd.read_csv(RESULTS_METRICS_DIR / "negative" / "final_filtered_vs_test.csv")
    df = pd.concat([df_pos, df_neg], ignore_index=True)
    df["class_label"] = df["target_label"].map({0: "Negativo", 1: "Positivo"})
    colors = df["target_label"].map({0: "#c0392b", 1: "#2e8b57"}).tolist()

    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
    axes = axes.ravel()

    axes[0].bar(df["class_label"], df["FID"], color=colors)
    axes[0].set_title("FID sul test - più basso è meglio")
    axes[0].grid(axis="y", alpha=0.25)

    axes[1].bar(df["class_label"], df["IS_mean"], yerr=df["IS_std"], color=colors, capsize=4)
    axes[1].set_title("Inception Score sul test")
    axes[1].grid(axis="y", alpha=0.25)

    prdc_metrics = ["precision", "recall", "density", "coverage"]
    x = np.arange(len(df))
    width = 0.8 / len(prdc_metrics)
    prdc_colors = ["#54A24B", "#E45756", "#72B7B2", "#B279A2"]
    for i, metric in enumerate(prdc_metrics):
        axes[2].bar(x + (i - 1.5) * width, df[metric], width=width, label=metric, color=prdc_colors[i])
    axes[2].set_xticks(x, df["class_label"])
    axes[2].set_title("PRDC sul test")
    axes[2].legend(ncol=2, fontsize=8)
    axes[2].grid(axis="y", alpha=0.25)

    table_df = df[["class_label", "FID", "IS_mean", *prdc_metrics]].round(4)
    axes[3].axis("off")
    table = axes[3].table(cellText=table_df.values, colLabels=table_df.columns, loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1.0, 1.25)

    fig.suptitle("Filtrate vs Test - Positivo vs Negativo", fontsize=14)
    output_path = RESULTS_PLOTS_DIR / "final_filtered_test_positivevsnegative.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Salvato:", output_path)


## 10. Stop manuale del processo


In [ ]:
# Cella lasciata inattiva per consentire Run All senza fermare il training.
# Per fermare manualmente un processo, decommenta le righe sotto e imposta il PID giusto.
# import os
# import signal
# os.kill(pid_ldm, signal.SIGTERM)
print("Stop manuale disattivato per Run All.")


## 11. Griglia reali vs sintetiche per slide

Cella di sola lettura, stessa logica del notebook `08_Showcase_Reali_vs_Sintetiche.ipynb`: confronta 4 reali (test, classe positiva) a sinistra con le 4 sintetiche a punteggio di filtro piu' alto a destra. Nessun calcolo nuovo.

In [ ]:
REAL_TEST_POSITIVE_DIR = DATA_PROCESSED_DIR / "test" / "1"
SYNTHETIC_FILTERED_DIR = DATA_DIR / "synthetic" / "06_ldm_extra1361_fromscratch" / "positive"

N_REAL = 4
N_SYNTH = 4

real_paths = sorted(REAL_TEST_POSITIVE_DIR.glob("*.png"))[:N_REAL]
synth_paths = sorted(SYNTHETIC_FILTERED_DIR.glob("synth_filtered_*.png"))[:N_SYNTH]

assert len(real_paths) == N_REAL, f"Solo {len(real_paths)} reali disponibili, richieste {N_REAL}."
assert len(synth_paths) == N_SYNTH, f"Solo {len(synth_paths)} sintetiche disponibili, richieste {N_SYNTH}."

print("Reali selezionate:")
for path in real_paths:
    print(" ", path.name)
print("Sintetiche selezionate:")
for path in synth_paths:
    print(" ", path.name)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage

GRID_ROWS = 2
GRID_COLS = 4  # colonne 0-1 = reali, colonne 2-3 = sintetiche

fig, axes = plt.subplots(GRID_ROWS, GRID_COLS, figsize=(12, 6.5))

for position, path in enumerate(real_paths):
    row, col = divmod(position, 2)
    with PILImage.open(path) as image:
        axes[row, col].imshow(np.asarray(image.convert("L")), cmap="gray", vmin=0, vmax=255)

for position, path in enumerate(synth_paths):
    row, col = divmod(position, 2)
    with PILImage.open(path) as image:
        axes[row, col + 2].imshow(np.asarray(image.convert("L")), cmap="gray", vmin=0, vmax=255)

for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])

fig.tight_layout(rect=(0, 0, 1, 0.90))

pos_left_a = axes[0, 0].get_position()
pos_left_b = axes[0, 1].get_position()
pos_right_a = axes[0, 2].get_position()
pos_right_b = axes[0, 3].get_position()
left_center_x = (pos_left_a.x0 + pos_left_b.x1) / 2
right_center_x = (pos_right_a.x0 + pos_right_b.x1) / 2
header_y = pos_left_a.y1 + 0.03

fig.text(left_center_x, header_y, "REALI - test set, classe positiva", ha="center", fontsize=13, fontweight="bold")
fig.text(right_center_x, header_y, "SINTETICHE - punteggio filtro piu' alto", ha="center", fontsize=13, fontweight="bold")
fig.suptitle(EXPERIMENT_NAME, fontsize=10, y=header_y + 0.07)

output_path = RESULTS_PLOTS_DIR / "real_vs_synthetic_grid_for_slide.png"
fig.savefig(output_path, dpi=200, bbox_inches="tight")
plt.show()

print("Griglia salvata in:", output_path)

## 12. Riepilogo artefatti

In [ ]:
from pathlib import Path


def human_size(n_bytes):
    n = float(n_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024 or unit == "TB":
            return f"{n:.1f} {unit}"
        n /= 1024


rows = []
for path in sorted(EXPERIMENT_DIR.rglob("*")):
    if path.is_file():
        rows.append(
            {
                "path": path.relative_to(EXPERIMENT_DIR).as_posix(),
                "size": human_size(path.stat().st_size),
            }
        )

summary_df = pd.DataFrame(rows)
print("File artefatti:", len(summary_df))
display(summary_df.head(80))
